# 2-Neuron Nonlinear Hawkes: Parameter Search (shared Poisson input, polynomial nonlinearity)

**Fix for 0-candidate problem:** adds a shared external Poisson input (rate `lambda_X`, amplitude `alpha_X`) to both neurons.  
This boosts `Var(v)` — and therefore the 1-loop mismatch Δ — without touching the stability eigenvalues.  
Nonlinearity: `phi(v, b, c) = (max(v,0) + c)^2 / b` so phi'' = 2/b is constant above threshold.

**Model (discrete-time Euler, symmetric)**:
```
v1[j+1] = max(0, v1[j] + (dt*(-v1[j]+mu) + alpha_self*spk1[j] + alpha_cross*spk2[j] + alpha_X*spkX[j]) / tau)
v2[j+1] = max(0, v2[j] + (dt*(-v2[j]+mu) + alpha_cross*spk1[j] + alpha_self*spk2[j] + alpha_X*spkX[j]) / tau)
spkX[j] ~ Bernoulli(lambda_X * dt)   # common Poisson input
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats.qmc import LatinHypercube, scale
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Nonlinearity: phi(v, b, c) = (max(v,0) + c)^2 / b
# phi'  = 2*(v+c)/b   for v > 0,  else 0
# phi'' = 2/b          for v > 0  (constant curvature — makes 1-loop correction clean)
# ---------------------------------------------------------------------------
def phi(v, b, c):
    return (np.maximum(v, 0.0) + c) ** 2 / b

def phi_p(v, b, c):
    """First derivative of phi."""
    return np.where(np.asarray(v) > 0.0, 2.0 * (np.asarray(v) + c) / b, 0.0)

def phi_pp(b):
    """Second derivative of phi above threshold (constant)."""
    return 2.0 / b

# ---------------------------------------------------------------------------
# Fixed-point solver
# Symmetric mean-field: 0 = (-v* + mu)/tau + (alpha_self+alpha_cross)*phi(v*)/tau + alpha_X*lambda_X/tau
#   => v* = mu + (alpha_self + alpha_cross)*phi(v*) + alpha_X*lambda_X
# ---------------------------------------------------------------------------
def find_fp(alpha_self, alpha_cross, alpha_X, lambda_X, mu, tau, b, c):
    """Return the unique positive fixed point, or None."""
    def rhs(v):
        return (-v + mu) / tau + (alpha_self + alpha_cross) * phi(v, b, c) / tau + alpha_X * lambda_X / tau

    vhi = max(mu * 30 + alpha_X * lambda_X * 30, 200.0)
    vs = np.linspace(1e-3, vhi, 10000)
    fs = rhs(vs)
    for i in range(len(vs) - 1):
        if fs[i] * fs[i + 1] < 0:
            try:
                vbar = brentq(rhs, vs[i], vs[i + 1], xtol=1e-12)
                if vbar > 0:
                    return vbar
            except Exception:
                pass
    return None

# ---------------------------------------------------------------------------
# Analytical quantities
# ---------------------------------------------------------------------------
def stability_eigenvalues(vstar, alpha_self, alpha_cross, tau, b, c):
    """Symmetric and anti-symmetric eigenvalues of the 2x2 Jacobian."""
    pp = float(phi_p(vstar, b, c))
    lam_sym  = (-1.0 + (alpha_self + alpha_cross) * pp) / tau
    lam_anti = (-1.0 + (alpha_self - alpha_cross) * pp) / tau
    return lam_sym, lam_anti

def predict_mismatch(vstar, alpha_self, alpha_cross, alpha_X, lambda_X, tau, b, c):
    """
    Diagonal Lyapunov / Jensen 1-loop correction.
    D_recurrent = (alpha_self^2 + alpha_cross^2) * phi(v*)
    D_common     = alpha_X^2 * lambda_X
    Var(v) approx D_total / (2 * |lambda_sym| * tau)
    Delta = phi''(v*) / (2*phi(v*)) * Var(v) = (1/b) * Var(v) / phi(v*)
    """
    lam_sym, _ = stability_eigenvalues(vstar, alpha_self, alpha_cross, tau, b, c)
    if lam_sym >= 0:
        return None
    phi_star   = float(phi(vstar, b, c))
    D_recurrent = (alpha_self**2 + alpha_cross**2) * phi_star
    D_common    = alpha_X**2 * lambda_X
    D_total     = D_recurrent + D_common
    var_v       = D_total / (2.0 * abs(lam_sym) * tau)
    delta       = (1.0 / b) * var_v / phi_star
    return delta, var_v, D_recurrent, D_common

print("Cell 1: Model definitions loaded.")
print(f"  phi(2.0,  b=5, c=2) = {phi(2.0, 5, 2):.5f}")
print(f"  phi_p(2.0, b=5, c=2) = {phi_p(2.0, 5, 2):.5f}")
print(f"  phi_pp(b=5)           = {phi_pp(5):.5f}")

In [ ]:
# ===========================================================================
# VERIFICATION: test one hardcoded parameter set BEFORE the big scan
# Expected to pass all filters.
# b=5, c=2, mu=4, tau=0.5, alpha_self=0.1, alpha_cross=0.1, alpha_X=1.0, lambda_X=20
# ===========================================================================
print("=" * 65)
print("VERIFICATION: hardcoded example")
print("=" * 65)

_b, _c, _mu, _tau = 5.0, 2.0, 4.0, 0.5
_as, _ac, _aX, _lX = 0.1, 0.1, 1.0, 20.0

_vstar = find_fp(_as, _ac, _aX, _lX, _mu, _tau, _b, _c)
print(f"\nParameters: b={_b}, c={_c}, mu={_mu}, tau={_tau}")
print(f"            alpha_self={_as}, alpha_cross={_ac}, alpha_X={_aX}, lambda_X={_lX}")
print(f"\nFilter 1 (FP > 0):       vstar = {_vstar}  =>  ", end="")
print("PASS" if (_vstar is not None and _vstar > 0) else "FAIL")

if _vstar is not None:
    _lam_sym, _lam_anti = stability_eigenvalues(_vstar, _as, _ac, _tau, _b, _c)
    print(f"\nFilter 2 (|lambda_sym| > 0.15): lambda_sym={_lam_sym:.4f}, |lambda_sym|={abs(_lam_sym):.4f}  =>  ", end="")
    print("PASS" if abs(_lam_sym) > 0.15 else "FAIL")

    print(f"Filter 3 (lambda_anti < 0):    lambda_anti={_lam_anti:.4f}  =>  ", end="")
    print("PASS" if _lam_anti < 0 else "FAIL")

    _phi_star = float(phi(_vstar, _b, _c))
    print(f"Filter 4 (5 < phi* < 500 Hz):  phi* = {_phi_star:.4f} Hz  =>  ", end="")
    print("PASS" if 5.0 < _phi_star < 500.0 else "FAIL")

    _res = predict_mismatch(_vstar, _as, _ac, _aX, _lX, _tau, _b, _c)
    if _res:
        _delta, _var_v, _D_rec, _D_com = _res
        _sdv = np.sqrt(_var_v)
        print(f"Filter 5 (v*/sigma > 3):       v*/sigma = {_vstar/_sdv:.4f}  =>  ", end="")
        print("PASS" if _vstar / _sdv > 3.0 else "FAIL")

        print(f"Filter 6 (0.02 < Delta < 0.20): Delta = {_delta:.5f}  =>  ", end="")
        print("PASS" if 0.02 < _delta < 0.20 else "FAIL")

        _jump = max(_as, _ac, _aX) / _tau
        print(f"Filter 7 (jump < 0.6*v*):      max_alpha/tau = {_jump:.4f},  0.6*v* = {0.6*_vstar:.4f}  =>  ", end="")
        print("PASS" if _jump < 0.6 * _vstar else "FAIL")

        _v_low = max(0.0, _vstar - 2.0 * _sdv)
        _phi_low = float(phi(_v_low, _b, _c))
        print(f"Filter 8 (phi(v*-2sigma) > 2): phi(v*-2sigma) = {_phi_low:.4f} Hz  =>  ", end="")
        print("PASS" if _phi_low > 2.0 else "FAIL")

        print(f"\n  Var(v)       = {_var_v:.5f}")
        print(f"  D_recurrent  = {_D_rec:.5f}")
        print(f"  D_common     = {_D_com:.5f}  ({100*_D_com/(_D_rec+_D_com):.1f}% of total)")
        print(f"  Delta        = {100*_delta:.3f}%")

print("\n" + "=" * 65)

In [ ]:
# ===========================================================================
# LHS SCAN: 100,000 samples over 8 parameters
# Parameters: b, c, mu, tau, alpha_self, alpha_cross, alpha_X, lambda_X
# ===========================================================================
N_SAMPLES = 100_000

BOUNDS_LOW  = np.array([ 0.5,  0.5,  1.0,  0.1,  0.0,   0.0,   0.1,   1.0])
BOUNDS_HIGH = np.array([20.0,  5.0, 10.0,  3.0,  1.0,   1.0,   3.0,  50.0])
PNAMES = ['b', 'c', 'mu', 'tau', 'alpha_self', 'alpha_cross', 'alpha_X', 'lambda_X']

print(f"Generating {N_SAMPLES:,} Latin Hypercube samples (8D)...")
sampler = LatinHypercube(d=8, seed=42)
raw = sampler.random(N_SAMPLES)
samples = scale(raw, BOUNDS_LOW, BOUNDS_HIGH)
print("Done. Running analytical filter scan...\n")

# ---------------------------------------------------------------------------
# Rejection counters (one per filter)
# ---------------------------------------------------------------------------
rej = [0] * 8   # one slot per filter
good_params = []

for i, s in enumerate(samples):
    b, c, mu, tau, a_self, a_cross, a_X, lX = (
        float(s[0]), float(s[1]), float(s[2]), float(s[3]),
        float(s[4]), float(s[5]), float(s[6]), float(s[7])
    )

    # Filter 1: fixed point must exist and be positive
    vstar = find_fp(a_self, a_cross, a_X, lX, mu, tau, b, c)
    if vstar is None or vstar <= 0.0:
        rej[0] += 1
        continue

    # Filter 2: stability margin |lambda_sym| > 0.15
    lam_sym, lam_anti = stability_eigenvalues(vstar, a_self, a_cross, tau, b, c)
    if abs(lam_sym) <= 0.15:
        rej[1] += 1
        continue

    # Filter 3: anti-symmetric mode stable
    if lam_anti >= 0.0:
        rej[2] += 1
        continue

    # Filter 4: firing rate 5 -- 500 Hz
    phi_star = float(phi(vstar, b, c))
    if phi_star < 5.0 or phi_star > 500.0:
        rej[3] += 1
        continue

    # Need mismatch for filters 5-8
    res = predict_mismatch(vstar, a_self, a_cross, a_X, lX, tau, b, c)
    if res is None:
        rej[4] += 1   # counts against filter 5 bucket (ill-conditioned)
        continue
    delta, var_v, D_rec, D_com = res
    sdv = np.sqrt(max(var_v, 1e-15))

    # Filter 5: voltage safety v* / sigma > 3
    if vstar / sdv <= 3.0:
        rej[4] += 1
        continue

    # Filter 6: mismatch target 0.02 < Delta < 0.20
    if not (0.02 < delta < 0.20):
        rej[5] += 1
        continue

    # Filter 7: jump condition (relaxed) max(alpha_self, alpha_cross, alpha_X)/tau < 0.6*v*
    if max(a_self, a_cross, a_X) / tau >= 0.6 * vstar:
        rej[6] += 1
        continue

    # Filter 8: consistent spiking at lower end phi(max(0, v*-2*sigma)) > 2 Hz
    v_low = max(0.0, vstar - 2.0 * sdv)
    if float(phi(v_low, b, c)) <= 2.0:
        rej[7] += 1
        continue

    good_params.append({
        'b': b, 'c': c, 'mu': mu, 'tau': tau,
        'alpha_self': a_self, 'alpha_cross': a_cross,
        'alpha_X': a_X, 'lambda_X': lX,
        'vstar': vstar, 'phi_star': phi_star,
        'delta': delta, 'var_v': var_v,
        'D_recurrent': D_rec, 'D_common': D_com,
        'lam_sym': lam_sym, 'lam_anti': lam_anti,
    })

# Sort by closeness to 7% mismatch
good_params.sort(key=lambda d: abs(d['delta'] - 0.07))

# ---------------------------------------------------------------------------
# Print rejection counts
# ---------------------------------------------------------------------------
filter_labels = [
    "Filter 1 (no FP or FP <= 0)",
    "Filter 2 (|lam_sym| <= 0.15)",
    "Filter 3 (lam_anti >= 0)",
    "Filter 4 (phi* out of [5,500] Hz)",
    "Filter 5 (v*/sigma <= 3 or ill-cond)",
    "Filter 6 (Delta out of [2%,20%])",
    "Filter 7 (jump condition violated)",
    "Filter 8 (phi at lower tail <= 2 Hz)",
]
print("-" * 55)
for label, count in zip(filter_labels, rej):
    print(f"  {label:<40}: {count:>7,}")
print("-" * 55)
print(f"  {'PASSED all filters':<40}: {len(good_params):>7,}")
print("-" * 55)

# ---------------------------------------------------------------------------
# Print top 20 candidates
# ---------------------------------------------------------------------------
print()
hdr = (f"{'#':>3}  {'b':>5} {'c':>5} {'mu':>5} {'tau':>6} "
       f"{'a_self':>7} {'a_cross':>8} {'a_X':>5} {'lX':>5} | "
       f"{'v*':>6} {'phi*(Hz)':>8} {'Delta%':>8} {'|lam_s|':>8}")
print("-" * len(hdr))
print(hdr)
print("-" * len(hdr))
for idx, p in enumerate(good_params[:20]):
    print(f"{idx+1:>3}  "
          f"{p['b']:>5.2f} {p['c']:>5.2f} {p['mu']:>5.2f} {p['tau']:>6.4f} "
          f"{p['alpha_self']:>7.4f} {p['alpha_cross']:>8.4f} "
          f"{p['alpha_X']:>5.2f} {p['lambda_X']:>5.1f} | "
          f"{p['vstar']:>6.3f} {p['phi_star']:>8.2f} "
          f"{100*p['delta']:>8.3f} {abs(p['lam_sym']):>8.4f}")

In [ ]:
# ===========================================================================
# STOCHASTIC SIMULATION of top 12 candidates
# ===========================================================================
N_CANDIDATES  = min(12, len(good_params))
N_TRIALS      = 8
T_TOTAL       = 3000.0   # seconds
T_BURNIN      = 500.0    # seconds
DT            = 5e-3     # time step
BLOWUP_THRESH = 200.0

N_STEPS  = int(T_TOTAL  / DT)
N_BURNIN = int(T_BURNIN / DT)

rng = np.random.default_rng(2024)
sim_results = []

if N_CANDIDATES == 0:
    print("No candidates to simulate — check filter counts above.")
else:
    print(f"Running stochastic simulations: {N_CANDIDATES} candidates x {N_TRIALS} trials")
    print(f"T={T_TOTAL}s, dt={DT}s, burnin={T_BURNIN}s, N_STEPS={N_STEPS:,}\n")

    for cand_idx in range(N_CANDIDATES):
        p       = good_params[cand_idx]
        b       = p['b'];     c      = p['c']
        mu      = p['mu'];    tau    = p['tau']
        a_self  = p['alpha_self'];  a_cross = p['alpha_cross']
        a_X     = p['alpha_X'];     lX      = p['lambda_X']
        vstar   = p['vstar'];       phi_star = p['phi_star']

        trial_rates1 = []; trial_rates2 = []
        trial_clamp1 = []; trial_clamp2 = []
        trial_var1   = []; trial_var2   = []
        blowup = False

        for trial in range(N_TRIALS):
            v1 = vstar; v2 = vstar
            sum_phi1 = 0.0; sum_phi2 = 0.0
            sum_v1sq = 0.0; sum_v1   = 0.0
            sum_v2sq = 0.0; sum_v2   = 0.0
            clamp1 = 0; clamp2 = 0
            n_post = 0
            blown  = False

            for j in range(N_STEPS):
                rate1 = float(phi(v1, b, c))
                rate2 = float(phi(v2, b, c))

                s1 = 1.0 if rng.random() < rate1 * DT else 0.0
                s2 = 1.0 if rng.random() < rate2 * DT else 0.0
                sX = 1.0 if rng.random() < lX    * DT else 0.0  # common input

                v1_new = v1 + (DT * (-v1 + mu) + a_self  * s1 + a_cross * s2 + a_X * sX) / tau
                v2_new = v2 + (DT * (-v2 + mu) + a_cross * s1 + a_self  * s2 + a_X * sX) / tau

                if v1_new < 0.0: clamp1 += 1; v1_new = 0.0
                if v2_new < 0.0: clamp2 += 1; v2_new = 0.0

                v1 = v1_new; v2 = v2_new

                if v1 > BLOWUP_THRESH or v2 > BLOWUP_THRESH or not (np.isfinite(v1) and np.isfinite(v2)):
                    blown = True; break

                if j >= N_BURNIN:
                    sum_phi1 += rate1; sum_phi2 += rate2
                    sum_v1   += v1;    sum_v2   += v2
                    sum_v1sq += v1*v1; sum_v2sq += v2*v2
                    n_post   += 1

            if blown:
                blowup = True; break

            if n_post > 0:
                trial_rates1.append(sum_phi1 / n_post)
                trial_rates2.append(sum_phi2 / n_post)
                n_rec = N_STEPS - N_BURNIN
                trial_clamp1.append(clamp1 / n_rec)
                trial_clamp2.append(clamp2 / n_rec)
                m1 = sum_v1 / n_post; m2 = sum_v2 / n_post
                trial_var1.append(sum_v1sq / n_post - m1**2)
                trial_var2.append(sum_v2sq / n_post - m2**2)

        if blowup or len(trial_rates1) == 0:
            sim_results.append({'cand_idx': cand_idx, 'status': 'UNSTABLE', **p})
            print(f"  Candidate {cand_idx+1:>2}: UNSTABLE (blowup)")
            continue

        mean_rate1  = float(np.mean(trial_rates1))
        mean_rate2  = float(np.mean(trial_rates2))
        mismatch1   = (mean_rate1 - phi_star) / phi_star
        mismatch2   = (mean_rate2 - phi_star) / phi_star
        fc1         = float(np.mean(trial_clamp1))
        fc2         = float(np.mean(trial_clamp2))
        meas_var1   = float(np.mean(trial_var1))
        meas_var2   = float(np.mean(trial_var2))

        clamp_ok = fc1 < 0.01 and fc2 < 0.01
        mismatch_ok = (0.03 <= abs(mismatch1) <= 0.15) and (0.03 <= abs(mismatch2) <= 0.15)
        valid = clamp_ok and mismatch_ok

        sim_results.append({
            'cand_idx': cand_idx, 'status': 'OK' if clamp_ok else 'CLAMP',
            'valid': valid,
            'mean_rate1': mean_rate1, 'mean_rate2': mean_rate2,
            'mf_rate': phi_star,
            'mismatch1': mismatch1, 'mismatch2': mismatch2,
            'frac_clamp1': fc1, 'frac_clamp2': fc2,
            'meas_var1': meas_var1, 'meas_var2': meas_var2,
            'trial_rates1': trial_rates1, 'trial_rates2': trial_rates2,
            **p
        })

        star = ' ★' if valid else ''
        print(f"  Candidate {cand_idx+1:>2}: "
              f"rate1={mean_rate1:6.2f} Hz (MF={phi_star:.2f})  "
              f"Δ1={100*mismatch1:+6.2f}%  "
              f"rate2={mean_rate2:6.2f} Hz  "
              f"Δ2={100*mismatch2:+6.2f}%  "
              f"clamp={100*fc1:.3f}%{star}")

In [ ]:
# ===========================================================================
# RESULTS TABLE sorted by |mismatch|
# ===========================================================================
ok_results = [
    r for r in sim_results
    if r.get('status') != 'UNSTABLE' and 'mismatch1' in r
]
ok_results.sort(key=lambda r: 0.5 * (abs(r['mismatch1']) + abs(r['mismatch2'])))

line = '=' * 145
print(line)
print(f"{'#':>3}  "
      f"{'b':>5} {'c':>5} {'mu':>5} {'tau':>6} "
      f"{'a_self':>7} {'a_cross':>8} {'a_X':>5} {'lX':>5} | "
      f"{'MF Hz':>7} {'rate1':>7} {'Δ1%':>7} "
      f"{'rate2':>7} {'Δ2%':>7} "
      f"{'clamp%':>7} {'valid':>6}")
print('-' * 145)
for rank, r in enumerate(ok_results):
    star = ' ★' if r.get('valid') else ''
    print(f"{rank+1:>3}  "
          f"{r['b']:>5.2f} {r['c']:>5.2f} {r['mu']:>5.2f} {r['tau']:>6.4f} "
          f"{r['alpha_self']:>7.4f} {r['alpha_cross']:>8.4f} "
          f"{r['alpha_X']:>5.2f} {r['lambda_X']:>5.1f} | "
          f"{r['mf_rate']:>7.2f} {r['mean_rate1']:>7.2f} {100*r['mismatch1']:>+7.2f} "
          f"{r['mean_rate2']:>7.2f} {100*r['mismatch2']:>+7.2f} "
          f"{100*r['frac_clamp1']:>7.4f}{star:>6}")

n_valid = sum(1 for r in ok_results if r.get('valid'))
print(line)
print(f"OK candidates: {len(ok_results)}  |  Valid (3-15% mismatch both neurons, clamp < 1%): {n_valid}")
print("Legend: ★ = |mismatch| in [3%, 15%] for BOTH neurons AND clamp fraction < 1%")

In [ ]:
# ===========================================================================
# 4-PANEL FIGURE for the best valid candidate
# ===========================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from scipy.ndimage import gaussian_filter1d

valid_results = [r for r in ok_results if r.get('valid')]
best = valid_results[0] if valid_results else (ok_results[0] if ok_results else None)

if best is None:
    print("No valid candidates found — no figure to produce.")
else:
    b       = best['b'];        c      = best['c']
    mu      = best['mu'];       tau    = best['tau']
    a_self  = best['alpha_self']; a_cross = best['alpha_cross']
    a_X     = best['alpha_X'];  lX      = best['lambda_X']
    vstar   = best['vstar'];    phi_star = best['mf_rate']

    # Noise budget
    D_rec = float((a_self**2 + a_cross**2) * phi_star)
    D_com = float(a_X**2 * lX)

    # Re-run one long trace for plotting
    DT_PLOT  = DT
    T_TRACE  = 3300.0
    N_TRACE  = int(T_TRACE / DT_PLOT)
    N_BURN   = int(500.0   / DT_PLOT)
    PLOT_WIN = int(300.0   / DT_PLOT)

    rng_plot = np.random.default_rng(999)
    v1_tr = np.zeros(N_TRACE); v2_tr = np.zeros(N_TRACE)
    phi1_tr = np.zeros(N_TRACE); phi2_tr = np.zeros(N_TRACE)

    v1 = vstar; v2 = vstar
    for j in range(N_TRACE):
        r1 = float(phi(v1, b, c)); r2 = float(phi(v2, b, c))
        s1 = 1.0 if rng_plot.random() < r1 * DT_PLOT else 0.0
        s2 = 1.0 if rng_plot.random() < r2 * DT_PLOT else 0.0
        sX = 1.0 if rng_plot.random() < lX * DT_PLOT else 0.0
        v1_new = v1 + (DT_PLOT * (-v1 + mu) + a_self  * s1 + a_cross * s2 + a_X * sX) / tau
        v2_new = v2 + (DT_PLOT * (-v2 + mu) + a_cross * s1 + a_self  * s2 + a_X * sX) / tau
        v1 = max(0.0, v1_new); v2 = max(0.0, v2_new)
        v1_tr[j] = v1; v2_tr[j] = v2
        phi1_tr[j] = r1; phi2_tr[j] = r2

    t_axis   = np.arange(N_TRACE) * DT_PLOT
    t_win    = t_axis[N_BURN: N_BURN + PLOT_WIN] - t_axis[N_BURN]
    v1_win   = v1_tr[N_BURN: N_BURN + PLOT_WIN]
    v2_win   = v2_tr[N_BURN: N_BURN + PLOT_WIN]
    phi1_win = phi1_tr[N_BURN: N_BURN + PLOT_WIN]
    phi2_win = phi2_tr[N_BURN: N_BURN + PLOT_WIN]

    sig_pts    = max(1, int(0.5 / DT_PLOT))
    phi1_sm    = gaussian_filter1d(phi1_win.astype(float), sig_pts)
    phi2_sm    = gaussian_filter1d(phi2_win.astype(float), sig_pts)

    # Phase portrait grid
    v_lo = max(0.01, vstar * 0.1)
    v_hi = vstar * 2.5
    v_range = np.linspace(v_lo, v_hi, 200)
    V1g, V2g = np.meshgrid(v_range, v_range)
    DV1g = (-(V1g - mu) + a_self  * phi(V1g, b, c) + a_cross * phi(V2g, b, c)) / tau
    DV2g = (-(V2g - mu) + a_cross * phi(V1g, b, c) + a_self  * phi(V2g, b, c)) / tau

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f'Best Valid Candidate\n'
        f'b={b:.2f}, c={c:.2f}, mu={mu:.2f}, tau={tau:.4f}s, '
        f'a_self={a_self:.4f}, a_cross={a_cross:.4f}, a_X={a_X:.3f}, lX={lX:.1f} Hz\n'
        f'MF rate={phi_star:.2f} Hz  |  '
        f'Sim rate1={best["mean_rate1"]:.2f} Hz ({100*best["mismatch1"]:+.1f}%)  '
        f'Sim rate2={best["mean_rate2"]:.2f} Hz ({100*best["mismatch2"]:+.1f}%)',
        fontsize=10
    )

    # Panel 1: voltage traces
    ax = axes[0, 0]
    ax.plot(t_win, v1_win, lw=0.4, alpha=0.7, color='steelblue', label='v1(t)')
    ax.plot(t_win, v2_win, lw=0.4, alpha=0.7, color='tomato',    label='v2(t)')
    ax.axhline(vstar, ls='--', color='navy', lw=1.5, label=f'v* = {vstar:.3f}')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Voltage v')
    ax.set_title('Stochastic voltage traces (300 s window)')
    ax.legend(fontsize=8)

    # Panel 2: smoothed firing rates
    ax = axes[0, 1]
    ax.plot(t_win, phi1_sm, lw=0.9, color='steelblue', label='rate1 (smoothed)')
    ax.plot(t_win, phi2_sm, lw=0.9, color='tomato',    label='rate2 (smoothed)')
    ax.axhline(phi_star,            ls='--', color='navy',      lw=1.5,
               label=f'MF rate = {phi_star:.2f} Hz')
    ax.axhline(best['mean_rate1'],  ls=':',  color='steelblue', lw=1.5,
               label=f'sim mean1 = {best["mean_rate1"]:.2f} Hz')
    ax.axhline(best['mean_rate2'],  ls=':',  color='tomato',    lw=1.5,
               label=f'sim mean2 = {best["mean_rate2"]:.2f} Hz')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Firing rate (Hz)')
    ax.set_title('Smoothed firing rates (sigma = 0.5 s)')
    ax.legend(fontsize=7)

    # Panel 3: joint histogram
    ax = axes[1, 0]
    v1_post = v1_tr[N_BURN:]; v2_post = v2_tr[N_BURN:]
    h, xe, ye = np.histogram2d(v1_post, v2_post, bins=60)
    ax.imshow(h.T, origin='lower', aspect='auto',
              extent=[xe[0], xe[-1], ye[0], ye[-1]], cmap='Blues')
    ax.plot(vstar, vstar, 'r*', ms=14, label=f'MF FP ({vstar:.3f}, {vstar:.3f})')
    ax.set_xlabel('v1'); ax.set_ylabel('v2')
    ax.set_title('Joint histogram of (v1, v2)')
    ax.legend(fontsize=8)

    # Panel 4: phase portrait + nullclines
    ax = axes[1, 1]
    skip = 10
    ax.quiver(V1g[::skip, ::skip], V2g[::skip, ::skip],
              DV1g[::skip, ::skip], DV2g[::skip, ::skip],
              alpha=0.4, width=0.003, color='gray')
    ax.contour(V1g, V2g, DV1g, levels=[0], colors=['steelblue'], linewidths=2)
    ax.contour(V1g, V2g, DV2g, levels=[0], colors=['tomato'],    linewidths=2)
    ax.plot(vstar, vstar, 'k*', ms=14, zorder=5)
    legend_els = [
        mlines.Line2D([], [], color='steelblue', lw=2, label='dv1/dt = 0'),
        mlines.Line2D([], [], color='tomato',    lw=2, label='dv2/dt = 0'),
        mlines.Line2D([], [], marker='*', color='k', ms=10, lw=0, label='Fixed point'),
    ]
    ax.legend(handles=legend_els, fontsize=8)
    ax.set_xlabel('v1'); ax.set_ylabel('v2')
    ax.set_title('Mean-field phase portrait + nullclines')

    plt.tight_layout()
    save_path = 'hawkes_2neuron_shared_input_best.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Figure saved to {save_path}")

    # -----------------------------------------------------------------------
    # Final summary printout
    # -----------------------------------------------------------------------
    lam_sym_best, lam_anti_best = stability_eigenvalues(
        vstar, a_self, a_cross, tau, b, c)
    pred_res = predict_mismatch(vstar, a_self, a_cross, a_X, lX, tau, b, c)
    pred_delta = pred_res[0] if pred_res else float('nan')
    frac_clamp = best['frac_clamp1']

    print()
    print("=" * 65)
    print("FINAL SUMMARY -- BEST VALID CANDIDATE")
    print("=" * 65)
    print(f"  b            = {b:.5f}")
    print(f"  c            = {c:.5f}")
    print(f"  mu           = {mu:.5f}")
    print(f"  tau          = {tau:.5f} s")
    print(f"  alpha_self   = {a_self:.5f}")
    print(f"  alpha_cross  = {a_cross:.5f}")
    print(f"  alpha_X      = {a_X:.5f}")
    print(f"  lambda_X     = {lX:.5f} Hz")
    print()
    print(f"  Fixed point:          v* = {vstar:.6f}")
    print(f"  MF firing rate:       phi(v*) = {phi_star:.5f} Hz")
    print(f"  Stability eigenvalues:")
    print(f"    lambda_sym  = {lam_sym_best:.5f}  (|lam_sym| = {abs(lam_sym_best):.5f})")
    print(f"    lambda_anti = {lam_anti_best:.5f}")
    print()
    print(f"  Predicted Delta   = {100*pred_delta:.4f}%")
    print(f"  Measured mismatch1 = {100*best['mismatch1']:+.4f}%")
    print(f"  Measured mismatch2 = {100*best['mismatch2']:+.4f}%")
    print()
    print(f"  Fraction of time at clamp = {100*frac_clamp:.5f}%  (should be ~0)")
    print()
    print(f"  Noise budget:")
    print(f"    D_common    (alpha_X^2 * lambda_X) = {D_com:.5f}")
    print(f"    D_recurrent (alpha^2  * phi*)       = {D_rec:.5f}")
    if D_rec + D_com > 0:
        print(f"    Common input fraction             = {100*D_com/(D_rec+D_com):.1f}%")
    print("=" * 65)

In [ ]:
# ===========================================================================
# CLEAN 2-PANEL FIGURE: voltage traces + firing rates with MF predictions
# Uses hardcoded best-candidate parameters so it runs standalone.
# ===========================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ── Best candidate parameters (paste from FINAL SUMMARY above) ───────────────
b        = 13.52923
c        = 0.65214
mu       = 1.45899
tau      = 2.42079
a_self   = 0.00996
a_cross  = 0.12709
a_X      = 2.01196
lX       = 5.49034

vstar    = 14.981148      # mean-field fixed point
phi_star = 18.06458       # mean-field firing rate (Hz)

# ── Simulation settings ───────────────────────────────────────────────────────
DT       = 5e-3           # s
T_TOTAL  = 2000.0         # total simulation time (s)
T_BURNIN = 100.0          # burn-in excluded from statistics (s)
T_WINDOW = 200.0          # plot window length (s)

N_STEPS  = int(T_TOTAL  / DT)
N_BURNIN = int(T_BURNIN / DT)
N_WINDOW = int(T_WINDOW / DT)

rng2 = np.random.default_rng(42)
v1_tr   = np.empty(N_STEPS)
v2_tr   = np.empty(N_STEPS)
phi1_tr = np.empty(N_STEPS)
phi2_tr = np.empty(N_STEPS)

v1 = vstar; v2 = vstar
for j in range(N_STEPS):
    r1 = float(phi(v1, b, c))
    r2 = float(phi(v2, b, c))
    s1 = 1.0 if rng2.random() < r1 * DT else 0.0
    s2 = 1.0 if rng2.random() < r2 * DT else 0.0
    sX = 1.0 if rng2.random() < lX * DT else 0.0
    v1_new = v1 + (DT * (-v1 + mu) + a_self  * s1 + a_cross * s2 + a_X * sX) / tau
    v2_new = v2 + (DT * (-v2 + mu) + a_cross * s1 + a_self  * s2 + a_X * sX) / tau
    v1 = max(0.0, v1_new); v2 = max(0.0, v2_new)
    v1_tr[j]   = v1;  v2_tr[j]   = v2
    phi1_tr[j] = r1;  phi2_tr[j] = r2

# ── Empirical statistics (full post-burnin trace) ────────────────────────────
v1_post   = v1_tr[N_BURNIN:];   v2_post   = v2_tr[N_BURNIN:]
phi1_post = phi1_tr[N_BURNIN:]; phi2_post = phi2_tr[N_BURNIN:]

emp_v1    = float(np.mean(v1_post));    emp_v2    = float(np.mean(v2_post))
emp_rate1 = float(np.mean(phi1_post));  emp_rate2 = float(np.mean(phi2_post))
emp_std1  = float(np.std(v1_post));     emp_std2  = float(np.std(v2_post))

# ── Extract plot window ───────────────────────────────────────────────────────
sl = slice(N_BURNIN, N_BURNIN + N_WINDOW)
t  = np.arange(N_WINDOW) * DT

v1_w    = v1_tr[sl];    v2_w    = v2_tr[sl]
phi1_w  = phi1_tr[sl];  phi2_w  = phi2_tr[sl]

sig = max(1, int(0.5 / DT))
phi1_sm = gaussian_filter1d(phi1_w, sig)
phi2_sm = gaussian_filter1d(phi2_w, sig)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.subplots_adjust(hspace=0.08)

# -- Panel 1: voltages --------------------------------------------------------
ax1.plot(t, v1_w, lw=0.5, alpha=0.65, color='#3A7EBF', label='Neuron 1  $v_1(t)$')
ax1.plot(t, v2_w, lw=0.5, alpha=0.65, color='#D95F5F', label='Neuron 2  $v_2(t)$')
ax1.axhline(vstar,  ls='--', lw=2.0, color='k',
            label=f'Mean-field  $v^*$ = {vstar:.3f}')
ax1.axhline(emp_v1, ls=':',  lw=1.8, color='#3A7EBF',
            label=f'Empirical $\\langle v_1 \\rangle$ = {emp_v1:.3f}')
ax1.axhline(emp_v2, ls=':',  lw=1.8, color='#D95F5F',
            label=f'Empirical $\\langle v_2 \\rangle$ = {emp_v2:.3f}')

ax1.set_ylabel('Voltage $v$', fontsize=13)
ax1.set_ylim(bottom=0)
ax1.legend(loc='upper right', fontsize=9, framealpha=0.85)
ax1.set_title(
    f'2-Neuron Nonlinear Hawkes  |  '
    f'$\\phi(v)=(v+c)^2/b$,  $b={b:.2f}$, $c={c:.3f}$,  '
    f'$\\mu={mu:.2f}$, $\\tau={tau:.2f}$ s,  '
    f'$\\alpha_X={a_X:.2f}$, $\\lambda_X={lX:.2f}$ Hz',
    fontsize=10
)
ax1.tick_params(labelbottom=False)

# -- Panel 2: firing rates ----------------------------------------------------
ax2.plot(t, phi1_sm, lw=1.2, color='#3A7EBF', label='Neuron 1  $\\phi(v_1)$ smoothed')
ax2.plot(t, phi2_sm, lw=1.2, color='#D95F5F', label='Neuron 2  $\\phi(v_2)$ smoothed')
ax2.axhline(phi_star,  ls='--', lw=2.2, color='k',
            label=f'Mean-field rate  {phi_star:.3f} Hz')
ax2.axhline(emp_rate1, ls=':',  lw=1.8, color='#3A7EBF',
            label=f'Empirical rate 1  {emp_rate1:.3f} Hz  ({100*(emp_rate1/phi_star-1):+.2f}%)')
ax2.axhline(emp_rate2, ls=':',  lw=1.8, color='#D95F5F',
            label=f'Empirical rate 2  {emp_rate2:.3f} Hz  ({100*(emp_rate2/phi_star-1):+.2f}%)')
ax2.axhspan(phi_star, 0.5*(emp_rate1 + emp_rate2),
            alpha=0.10, color='gray', label='MF → simulation gap')

ax2.set_xlabel('Time (s)', fontsize=13)
ax2.set_ylabel('Firing rate (Hz)', fontsize=13)
ax2.legend(loc='upper right', fontsize=9, framealpha=0.85)

plt.savefig('hawkes_2neuron_traces_clean.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: hawkes_2neuron_traces_clean.png")

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("=" * 58)
print("  MEAN-FIELD vs SIMULATION SUMMARY")
print(f"  (T = {T_TOTAL:.0f} s,  burn-in = {T_BURNIN:.0f} s,  dt = {DT} s)")
print("=" * 58)
print(f"  {'Quantity':<30} {'MF pred':>10} {'Neuron 1':>10} {'Neuron 2':>10}")
print("-" * 58)
print(f"  {'Fixed point / mean voltage':<30} {vstar:>10.4f} {emp_v1:>10.4f} {emp_v2:>10.4f}")
print(f"  {'Voltage std dev':<30} {'—':>10} {emp_std1:>10.4f} {emp_std2:>10.4f}")
print(f"  {'Firing rate (Hz)':<30} {phi_star:>10.4f} {emp_rate1:>10.4f} {emp_rate2:>10.4f}")
print(f"  {'Rate mismatch (%)':<30} {'0.0000':>10} {100*(emp_rate1/phi_star-1):>+10.4f} {100*(emp_rate2/phi_star-1):>+10.4f}")
print("=" * 58)